In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

base_dir = os.path.dirname(os.path.dirname(os.getcwd()))
data_dir = os.path.join(base_dir, 'data', 'processed_data', 'House2_full.csv')
sys.path.append(base_dir)

from src.tools.window_shifter import WindowShifter
from src.metrics.energy_base_metrics import MAE, NEP, Precision_energy_based, Recall_energy_based, F1_energy_based

In [5]:
df = pd.read_csv(data_dir, encoding = 'utf-8')
app_cols = ['Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']
invalid_mask = df['Aggregate'] < df[app_cols].sum(axis=1)
df.loc[invalid_mask, app_cols] = np.nan
df[app_cols] = df[app_cols].interpolate(method='linear').bfill().ffill()
df.loc[invalid_mask, 'Aggregate'] = np.nan
df['Aggregate'] = df['Aggregate'].interpolate(method='linear').bfill().ffill()
df = WindowShifter.shift(df, 50)
aggregate_cols = [i for i in df.columns[:-9] if 'Time' not in i]
df['remain'] = df['Aggregate_t0'] - df[app_cols].sum(axis = 1)
df['remain'] = df['remain'].clip(lower=0)  

In [6]:
df_time_to_date = pd.to_datetime(df['Time_t0'], format='%Y-%m-%d %H:%M:%S')
df['dow'] = df_time_to_date.dt.day_of_week
df['dom'] = df_time_to_date.dt.day
df['hour'] = df_time_to_date.dt.hour

In [7]:
df.head()

,Time_t0,Aggregate_t0,Time_t1,Aggregate_t1,Time_t2,Aggregate_t2,Time_t3,Aggregate_t3,Time_t4,Aggregate_t4,...,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9,remain,dow,dom,hour
49,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,2013-09-17 22:13:20,698.0,2013-09-17 22:13:17,696.0,...,0.0,0.0,0.0,0.0,0.0,0.0,609.0,1,17,22
50,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,2013-09-17 22:13:20,698.0,...,0.0,0.0,0.0,0.0,0.0,0.0,597.0,1,17,22
51,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,...,0.0,0.0,0.0,0.0,0.0,0.0,601.0,1,17,22
52,2013-09-17 22:14:01,691.0,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,...,0.0,0.0,0.0,0.0,0.0,0.0,606.0,1,17,22
53,2013-09-17 22:14:08,691.0,2013-09-17 22:14:01,691.0,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,...,0.0,0.0,0.0,0.0,0.0,0.0,606.0,1,17,22


In [9]:
df[app_cols].describe()

,Appliance1,Appliance2,Appliance3,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9
count,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06,5.733477e+06
mean,3.562417e+01,1.803707e+01,6.058062e+01,3.951857e+00,2.810471e+00,1.449910e+00,1.728182e+00,2.198316e+01,3.663726e-01
std,4.353730e+01,1.734733e+02,3.487188e+02,1.260141e+01,5.320088e+01,3.633231e+01,5.080037e+00,2.407520e+02,5.080304e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,8.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,1.639000e+03,2.555000e+03,2.480000e+03,5.120000e+02,1.251000e+03,1.024000e+03,1.536000e+03,2.939000e+03,3.840000e+02


### Ý tưởng ###
* Tạo 2 model 1 model để xem liệu thiết bị có đang hoạt động không nếu thiết bị đang hoạt động thì dự đoán công suất của nó

In [10]:
on_threshold = {
    'Appliance1': 15,    # Fridge-Freezer: may nen chay ~50-150W
    'Appliance2': 20,    # Washing Machine: nhieu che do tu 20W
    'Appliance3': 20,    # Dishwasher: che do nhe tu 10W
    'Appliance4': 15,    # Television: xem TV 50-200W
    'Appliance5': 50,   # Microwave: cong suat cao, standby rat thap
    'Appliance6': 50,   # Toaster: cong suat cao, bat cuc ngan
    'Appliance7': 10,    # Hi-Fi: cong suat thap
    'Appliance8': 100,   # Kettle: cong suat rat cao 2000-3000W
    'Appliance9': 5,    # Oven Extractor Fan: quat hut
}

In [11]:
x_cols = ['dow', 'dom', 'hour'] + aggregate_cols
y_cols = app_cols + ['remain']

In [16]:
df_classifier = df.copy()
for i in app_cols:
    df_classifier[i] = df_classifier[i]<on_threshold[i]
df_classifier.head()

,Time_t0,Aggregate_t0,Time_t1,Aggregate_t1,Time_t2,Aggregate_t2,Time_t3,Aggregate_t3,Time_t4,Aggregate_t4,...,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9,remain,dow,dom,hour
49,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,2013-09-17 22:13:20,698.0,2013-09-17 22:13:17,696.0,...,True,True,True,True,True,True,609.0,1,17,22
50,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,2013-09-17 22:13:20,698.0,...,True,True,True,True,True,True,597.0,1,17,22
51,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,2013-09-17 22:13:24,702.0,...,True,True,True,True,True,True,601.0,1,17,22
52,2013-09-17 22:14:01,691.0,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,2013-09-17 22:13:32,694.0,...,True,True,True,True,True,True,606.0,1,17,22
53,2013-09-17 22:14:08,691.0,2013-09-17 22:14:01,691.0,2013-09-17 22:13:54,686.0,2013-09-17 22:13:46,683.0,2013-09-17 22:13:39,695.0,...,True,True,True,True,True,True,606.0,1,17,22


In [29]:
from sklearn.model_selection import train_test_split
X_train_class, y_train_class = df_classifier[((df_classifier['dom']-1)//7+1)%2==0][x_cols], df_classifier[((df_classifier['dom']-1)//7+1)%2==0][y_cols]
X_train_class, X_val_class, y_train_class, y_val_class = train_test_split(X_train_class, y_train_class, test_size = 0.01)

In [36]:
df.loc[3953788][app_cols]

Appliance1     1.0
Appliance2     0.0
Appliance3     0.0
Appliance4    40.0
Appliance5     0.0
Appliance6     0.0
Appliance7     0.0
Appliance8     0.0
Appliance9     0.0
Name: 3953788, dtype: object

In [47]:
df.loc[3953788]['remain']

np.float64(10203.0)

In [51]:
app1_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app1_class.fit(X_train_class, y_train_class['Appliance1'],
    eval_set=[(X_val_class, y_val_class['Appliance1'])], 
    verbose=100)

app2_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app2_class.fit(X_train_class, y_train_class['Appliance2'],
    eval_set=[(X_val_class, y_val_class['Appliance2'])], 
    verbose=100)

app3_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app3_class.fit(X_train_class, y_train_class['Appliance3'],
    eval_set=[(X_val_class, y_val_class['Appliance3'])], 
    verbose=100)

app4_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app4_class.fit(X_train_class, y_train_class['Appliance4'],
    eval_set=[(X_val_class, y_val_class['Appliance4'])], 
    verbose=100)

app5_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app5_class.fit(X_train_class, y_train_class['Appliance5'],
    eval_set=[(X_val_class, y_val_class['Appliance5'])], 
    verbose=100)

app6_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app6_class.fit(X_train_class, y_train_class['Appliance6'],
    eval_set=[(X_val_class, y_val_class['Appliance6'])], 
    verbose=100)

app7_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app7_class.fit(X_train_class, y_train_class['Appliance7'],
    eval_set=[(X_val_class, y_val_class['Appliance7'])], 
    verbose=100)

app8_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app8_class.fit(X_train_class, y_train_class['Appliance8'],
    eval_set=[(X_val_class, y_val_class['Appliance8'])], 
    verbose=100)

app9_class = xgb.XGBClassifier(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
app9_class.fit(X_train_class, y_train_class['Appliance9'],
    eval_set=[(X_val_class, y_val_class['Appliance9'])], 
    verbose=100)

[0]	validation_0-logloss:0.65683
[100]	validation_0-logloss:0.36317
[200]	validation_0-logloss:0.33642
[300]	validation_0-logloss:0.31871
[400]	validation_0-logloss:0.30376
[500]	validation_0-logloss:0.29311
[600]	validation_0-logloss:0.28526
[700]	validation_0-logloss:0.27823
[800]	validation_0-logloss:0.27128
[900]	validation_0-logloss:0.26501
[1000]	validation_0-logloss:0.25928
[1100]	validation_0-logloss:0.25518
[1200]	validation_0-logloss:0.25049
[1300]	validation_0-logloss:0.24633
[1400]	validation_0-logloss:0.24278
[1500]	validation_0-logloss:0.23930
[1600]	validation_0-logloss:0.23501
[1700]	validation_0-logloss:0.23188
[1800]	validation_0-logloss:0.22875
[1900]	validation_0-logloss:0.22610
[2000]	validation_0-logloss:0.22384
[2100]	validation_0-logloss:0.22058
[2200]	validation_0-logloss:0.21688
[2300]	validation_0-logloss:0.21460
[2400]	validation_0-logloss:0.21196
[2500]	validation_0-logloss:0.20935
[2600]	validation_0-logloss:0.20717
[2700]	validation_0-logloss:0.20457
[280

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [52]:
X_test_class, y_test_class = df_classifier[((df_classifier['dom']-1)//7+1)%2==1][x_cols], df_classifier[((df_classifier['dom']-1)//7+1)%2==1][y_cols]

In [66]:
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix
f1_score(app1_class.predict(X_test_class), y_test_class['Appliance1'])

0.8420683940882188

In [67]:
f1_score(app2_class.predict(X_test_class), y_test_class['Appliance2'])

0.9837038039609947

In [68]:
f1_score(app3_class.predict(X_test_class), y_test_class['Appliance3'])

0.957349553847406

In [69]:
f1_score(app4_class.predict(X_test_class), y_test_class['Appliance4'])

0.9447709381619954

In [70]:
f1_score(app5_class.predict(X_test_class), y_test_class['Appliance5'])

0.9990212673745059

In [71]:
f1_score(app6_class.predict(X_test_class), y_test_class['Appliance6'])

0.9994341690537486

In [72]:
f1_score(app7_class.predict(X_test_class), y_test_class['Appliance7'])

0.9066264188254787

In [73]:
f1_score(app8_class.predict(X_test_class), y_test_class['Appliance8'])

0.9987439092525638

In [74]:
f1_score(app9_class.predict(X_test_class), y_test_class['Appliance9'])

0.9974003092913981

In [75]:
X_train, y_train, X_test, y_test = df[((df['dom']-1)//7+1)%2==0][x_cols], df[((df['dom']-1)//7+1)%2==0][y_cols], df[((df['dom']-1)//7+1)%2==1][x_cols], df[((df['dom']-1)//7+1)%2==1][y_cols]
X_test = X_test[:70000]
y_test = y_test[:70000]

In [76]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.05)

In [80]:
classifier = {
    'Appliance1': app1_class,
    'Appliance2': app2_class,
    'Appliance3': app3_class,
    'Appliance4': app4_class,
    'Appliance5': app5_class,
    'Appliance6': app6_class,
    'Appliance7': app7_class,
    'Appliance8': app8_class,
    'Appliance9': app9_class
}

In [78]:
regressor = {}

In [86]:
for app in app_cols:
    model = xgb.XGBRegressor(n_estimators=5000,      
    learning_rate=0.05,
    early_stopping_rounds=50, 
    tree_method='hist',      
    device="cuda")
    model.fit(X_train, y_train[app], eval_set=[(X_val, y_val[app])], verbose=100)
    regressor[app] = model

[0]	validation_0-rmse:43.05305
[100]	validation_0-rmse:31.92725
[200]	validation_0-rmse:30.58624
[300]	validation_0-rmse:29.87585
[400]	validation_0-rmse:29.27717
[500]	validation_0-rmse:28.80916
[600]	validation_0-rmse:28.37066
[700]	validation_0-rmse:28.00750
[800]	validation_0-rmse:27.73262
[900]	validation_0-rmse:27.45614
[1000]	validation_0-rmse:27.17491
[1100]	validation_0-rmse:26.98730
[1200]	validation_0-rmse:26.74218
[1300]	validation_0-rmse:26.57300
[1400]	validation_0-rmse:26.37680
[1500]	validation_0-rmse:26.12608
[1600]	validation_0-rmse:25.93647
[1700]	validation_0-rmse:25.75547
[1800]	validation_0-rmse:25.56315
[1900]	validation_0-rmse:25.39690
[2000]	validation_0-rmse:25.29172
[2100]	validation_0-rmse:25.15685
[2200]	validation_0-rmse:25.01523
[2300]	validation_0-rmse:24.84418
[2400]	validation_0-rmse:24.74371
[2500]	validation_0-rmse:24.59922
[2600]	validation_0-rmse:24.43812
[2700]	validation_0-rmse:24.32739
[2800]	validation_0-rmse:24.21107
[2900]	validation_0-rmse:2

In [107]:
pred = []
for app in app_cols:
    pred.append(regressor[app].predict(X_test))

In [112]:
def Precision_energy_based_fixed(y_predict, y_target, alpha = 1e-6):
    assert y_predict.shape[0] == y_target.shape[0], 'samples number not match'
    assert y_predict.shape[1] == y_target.shape[1], 'number of appliances not match'
    denominator = y_predict.sum(axis = 0) 
    numerator = np.minimum(y_predict, y_target).sum(axis = 0) 
    return (numerator + alpha) / (denominator + alpha)
def Recall_energy_based_fixed(y_predict, y_target, alpha = 1e-6):
    assert y_predict.shape[0] == y_target.shape[0], 'samples number not match'
    assert y_predict.shape[1] == y_target.shape[1], 'number of appliances not match'
    denominator = y_target.sum(axis = 0)
    numerator = np.minimum(y_predict, y_target).sum(axis = 0)
    return (numerator + alpha) / (denominator + alpha)
def F1_energy_based_fixed(y_predict, y_target, alpha = 1e-6):
    assert y_predict.shape[0] == y_target.shape[0], 'samples number not match'
    assert y_predict.shape[1] == y_target.shape[1], 'number of appliances not match'
    numerator = np.minimum(y_predict, y_target).sum(axis = 0)
    p_denominator = y_predict.sum(axis = 0)
    r_denominator = y_target.sum(axis = 0)
    precision = (numerator + alpha) / (p_denominator + alpha)
    recall = (numerator + alpha) / (r_denominator + alpha)
    return 2 * precision * recall / (precision + recall)
def NEP_fixed(y_predict, y_target, alpha = 1e-6):
    assert y_predict.shape[0] == y_target.shape[0], 'samples number not match'
    assert y_predict.shape[1] == y_target.shape[1], 'number of appliances not match'
    denominator = y_target.sum(axis = 0)
    numerator = np.abs(y_target - y_predict).sum(axis = 0)
    return (numerator + alpha) / (denominator + alpha)

In [116]:
overall_pred = pd.DataFrame(pred).transpose()
overall_pred

,0,1,2,3,4,5,6,7,8
0,20.973866,8.070931,8.358551,7.070478,0.026441,-0.649233,6.697180,0.243353,-1.959019
1,14.283939,1.218697,7.330341,6.438416,-17.489367,-0.045115,6.679277,-0.080858,-1.991928
2,9.927193,2.505585,6.157059,6.182587,-0.186359,0.912720,6.489318,0.681019,-1.980535
3,16.033241,12.257490,7.995152,8.645664,1.038141,1.382908,6.860340,0.268932,-2.348862
4,18.186836,6.091258,6.010300,10.150900,0.870025,0.770120,6.049146,0.293678,-2.128759
...,...,...,...,...,...,...,...,...,...
69995,53.749298,9.188641,3.144482,0.807602,-0.033690,-0.050292,-0.887254,-0.178946,-0.057002
69996,55.388710,8.439578,3.727341,1.220609,-0.033690,-0.031499,-0.408772,-0.194483,-0.038330
69997,27.625565,6.427782,3.156006,-0.606904,-0.033690,-0.023512,-0.458074,-0.201773,-0.037447
69998,11.904821,215.663467,1689.061401,-1.249819,14.911163,138.630234,3.005446,-132.509369,-1.970449


In [117]:
overall_pred = overall_pred.clip(lower=0)

In [118]:
F1_energy_based_fixed(overall_pred.to_numpy(), y_test.reset_index(drop = True).drop(columns = ['remain']).to_numpy()).mean()

np.float64(0.4496528031596347)

In [119]:
Precision_energy_based_fixed(overall_pred.to_numpy(), y_test.reset_index(drop = True).drop(columns = ['remain']).to_numpy()).mean()

np.float64(0.4596769670990869)

In [120]:
Recall_energy_based_fixed(overall_pred.to_numpy(), y_test.reset_index(drop = True).drop(columns = ['remain']).to_numpy()).mean()

np.float64(0.466026959457465)

In [121]:
NEP_fixed(overall_pred.to_numpy(), y_test.reset_index(drop = True).drop(columns = ['remain']).to_numpy()).mean()

np.float64(1.440196969855692)